# Phase 1~3: 학습 + 추론 + 앙상블 + 제출

All-in-One 전략: 처음부터 풀 스펙 모델을 만들고, backbone만 바꿔서 복제 학습 후 앙상블

**실행 흐름:**
1. Config 설정 (backbone 변경은 여기서만)
2. Dataset + Augmentation (도메인 적응형 풀 스펙)
3. Model (Dual-Stream Late Fusion)
4. Train (5-Fold CV + KD + Mixup)
5. Inference (TTA)
6. 앙상블 + Temperature Scaling
7. 제출 CSV 생성

## 0. Imports + Config

In [1]:
import os
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3080
VRAM: 10.0 GB


In [2]:
# ============================================================
# CONFIG — backbone 교체 시 여기만 변경
# ============================================================
@dataclass
class Config:
    # === 이것만 바꾸면 다른 모델 학습 가능 ===
    backbone: str = 'convnextv2_base.fcmae_ft_in22k_in1k_384'
    exp_name: str = 'convnextv2_base'  # 저장 폴더명
    img_size: int = 384

    # Training
    epochs: int = 30
    batch_size: int = 8
    grad_accum: int = 4  # effective batch = 32
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 0.01
    warmup_epochs: int = 2
    early_stopping_patience: int = 7

    # Regularization
    label_smoothing: float = 0.05
    mixup_alpha: float = 0.3
    cutmix_alpha: float = 1.0
    mix_prob: float = 0.5  # mixup/cutmix 적용 확률
    drop_path_rate: float = 0.2

    # Knowledge Distillation
    use_kd: bool = True
    kd_alpha: float = 0.3  # soft label 비중
    kd_temperature: float = 3.0

    # Fold
    n_folds: int = 5
    seed: int = 42

    # TTA
    tta_count: int = 5

    # Paths
    data_dir: str = '../data'
    output_dir: str = '../outputs'

cfg = Config()
seed_everything(cfg.seed)

# Output directory
exp_dir = Path(cfg.output_dir) / cfg.exp_name
exp_dir.mkdir(parents=True, exist_ok=True)
print(f'Experiment: {cfg.exp_name}')
print(f'Backbone: {cfg.backbone}')
print(f'Output: {exp_dir}')

Experiment: convnextv2_base
Backbone: convnextv2_base.fcmae_ft_in22k_in1k_384
Output: ..\outputs\convnextv2_base


## 1. Dataset + Augmentation

In [3]:
# 데이터 로딩
data_dir = Path(cfg.data_dir)
train_df = pd.read_csv(data_dir / 'train.csv')
dev_df = pd.read_csv(data_dir / 'dev.csv')
test_df = pd.read_csv(data_dir / 'sample_submission.csv')

# Train + Dev 통합
train_df['split'] = 'train'
dev_df['split'] = 'dev'
all_df = pd.concat([train_df, dev_df], ignore_index=True)
all_df['label_int'] = (all_df['label'] == 'unstable').astype(int)

# Soft label 로딩 (train에만 존재)
soft_labels_path = data_dir / 'soft_labels.csv'
if soft_labels_path.exists():
    soft_df = pd.read_csv(soft_labels_path)
    all_df = all_df.merge(soft_df[['id', 'soft_unstable_prob']], on='id', how='left')
    # Dev는 soft label 없음 → hard label 사용
    all_df['soft_unstable_prob'] = all_df['soft_unstable_prob'].fillna(
        all_df['label_int'].astype(float)
    )
    print(f'Soft labels loaded: {soft_df.shape[0]} samples')
else:
    all_df['soft_unstable_prob'] = all_df['label_int'].astype(float)
    print('WARNING: soft_labels.csv not found, using hard labels')

print(f'Total samples: {len(all_df)} (train: {len(train_df)}, dev: {len(dev_df)})')
print(f'Label distribution: {all_df["label"].value_counts().to_dict()}')

Soft labels loaded: 1000 samples
Total samples: 1100 (train: 1000, dev: 100)
Label distribution: {'unstable': 552, 'stable': 548}


In [4]:
def get_train_transforms(img_size):
    """도메인 적응형 풀 스펙 augmentation"""
    return A.Compose([
        A.Resize(img_size, img_size),

        # 조명 변동 (Train→Dev/Test 도메인 갭 해소)
        # EDA: Train brightness=214, Dev=191 → 밝기를 낮추는 방향
        A.RandomBrightnessContrast(
            brightness_limit=(-0.3, 0.1),  # 밝기 낮추는 쪽 강조
            contrast_limit=(-0.3, 0.3),
            p=0.8
        ),
        A.ColorJitter(
            brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6
        ),
        A.RandomGamma(gamma_limit=(70, 130), p=0.4),
        A.RandomToneCurve(scale=0.2, p=0.3),

        # 카메라 시점 변동
        A.Perspective(scale=(0.02, 0.08), p=0.5),
        A.Affine(
            scale=(0.85, 1.15),
            translate_percent=(-0.1, 0.1),
            rotate=(-15, 15),
            shear=(-10, 10),
            p=0.6
        ),

        # 기본 증강
        A.HorizontalFlip(p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.2),

        # 정규화
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size):
    """Test-Time Augmentation: 가벼운 변형들"""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

In [5]:
class StructuralDataset(Dataset):
    def __init__(self, df, data_dir, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = row['id']

        # 이미지 경로 결정
        if self.is_test:
            base = self.data_dir / 'test' / sample_id
        elif row.get('split', 'train') == 'dev':
            base = self.data_dir / 'dev' / sample_id
        else:
            base = self.data_dir / 'train' / sample_id

        front = cv2.imread(str(base / 'front.png'))
        front = cv2.cvtColor(front, cv2.COLOR_BGR2RGB)
        top = cv2.imread(str(base / 'top.png'))
        top = cv2.cvtColor(top, cv2.COLOR_BGR2RGB)

        if self.transforms:
            # front와 top에 동일한 기하 변환 적용을 위해 같이 증강
            # ReplayCompose로 동일 변환 보장
            aug_front = self.transforms(image=front)
            aug_top = self.transforms(image=top)
            front = aug_front['image']
            top = aug_top['image']

        result = {'front': front, 'top': top, 'id': sample_id}

        if not self.is_test:
            result['label'] = int(row['label_int'])
            result['soft_label'] = float(row.get('soft_unstable_prob', row['label_int']))

        return result

# 테스트
test_ds = StructuralDataset(
    all_df.head(2), data_dir, get_train_transforms(cfg.img_size)
)
sample = test_ds[0]
print(f'Front shape: {sample["front"].shape}')
print(f'Top shape: {sample["top"].shape}')
print(f'Label: {sample["label"]}, Soft: {sample["soft_label"]:.4f}')

Front shape: torch.Size([3, 384, 384])
Top shape: torch.Size([3, 384, 384])
Label: 1, Soft: 0.9993


## 2. Model

In [6]:
class DualStreamModel(nn.Module):
    def __init__(self, backbone_name, num_classes=2, drop_path_rate=0.2):
        super().__init__()
        self.backbone_front = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )
        self.backbone_top = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )

        # Gradient checkpointing
        if hasattr(self.backbone_front, 'set_grad_checkpointing'):
            self.backbone_front.set_grad_checkpointing(True)
            self.backbone_top.set_grad_checkpointing(True)

        feat_dim = self.backbone_front.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        self.num_classes = num_classes

    def forward(self, front, top):
        feat_front = self.backbone_front(front)
        feat_top = self.backbone_top(top)
        combined = torch.cat([feat_front, feat_top], dim=1)
        logits = self.head(combined)
        return logits

# 모델 생성 테스트
model = DualStreamModel(cfg.backbone, drop_path_rate=cfg.drop_path_rate)
feat_dim = model.backbone_front.num_features
total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Feature dim: {feat_dim}')
print(f'Total params: {total_params:.1f}M')
print(f'Trainable params: {trainable_params:.1f}M')
del model
torch.cuda.empty_cache()

Feature dim: 1024
Total params: 176.5M
Trainable params: 176.5M


## 3. Training Utilities

In [7]:
def mixup_data(front, top, labels, soft_labels, alpha=0.3):
    """Mixup augmentation on batch"""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = front.size(0)
    index = torch.randperm(batch_size).to(front.device)

    mixed_front = lam * front + (1 - lam) * front[index]
    mixed_top = lam * top + (1 - lam) * top[index]
    labels_a, labels_b = labels, labels[index]
    soft_a, soft_b = soft_labels, soft_labels[index]
    return mixed_front, mixed_top, labels_a, labels_b, soft_a, soft_b, lam


def cutmix_data(front, top, labels, soft_labels, alpha=1.0):
    """CutMix augmentation on batch"""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = front.size(0)
    index = torch.randperm(batch_size).to(front.device)

    _, _, H, W = front.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    mixed_front = front.clone()
    mixed_front[:, :, y1:y2, x1:x2] = front[index, :, y1:y2, x1:x2]
    mixed_top = top.clone()
    mixed_top[:, :, y1:y2, x1:x2] = top[index, :, y1:y2, x1:x2]

    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    labels_a, labels_b = labels, labels[index]
    soft_a, soft_b = soft_labels, soft_labels[index]
    return mixed_front, mixed_top, labels_a, labels_b, soft_a, soft_b, lam


def compute_loss(logits, labels_a, labels_b, soft_a, soft_b, lam, cfg):
    """KD + Label Smoothing + Mixup/CutMix 통합 loss"""
    ce_loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

    # Hard label loss (with mixup)
    hard_loss = lam * ce_loss_fn(logits, labels_a) + (1 - lam) * ce_loss_fn(logits, labels_b)

    if cfg.use_kd:
        # Soft label (KD) loss
        T = cfg.kd_temperature
        soft_targets_a = torch.stack([1 - soft_a, soft_a], dim=1)  # [stable, unstable]
        soft_targets_b = torch.stack([1 - soft_b, soft_b], dim=1)
        soft_targets = lam * soft_targets_a + (1 - lam) * soft_targets_b

        log_probs = F.log_softmax(logits / T, dim=1)
        soft_targets_scaled = F.softmax(soft_targets / T, dim=1)
        kd_loss = F.kl_div(log_probs, soft_targets_scaled, reduction='batchmean') * (T * T)

        total_loss = (1 - cfg.kd_alpha) * hard_loss + cfg.kd_alpha * kd_loss
    else:
        total_loss = hard_loss

    return total_loss


class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, steps_per_epoch):
        self.optimizer = optimizer
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_steps = total_epochs * steps_per_epoch
        self.current_step = 0
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            # Linear warmup
            scale = self.current_step / self.warmup_steps
        else:
            # Cosine decay
            progress = (self.current_step - self.warmup_steps) / (
                self.total_steps - self.warmup_steps
            )
            scale = 0.5 * (1 + np.cos(np.pi * progress))

        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

## 4. Train Loop (5-Fold CV)

In [8]:
def train_one_fold(fold, train_idx, val_idx, all_df, cfg):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}')
    print(f'{"="*60}')

    train_data = all_df.iloc[train_idx]
    val_data = all_df.iloc[val_idx]
    print(f'Train: {len(train_data)} | Val: {len(val_data)}')
    print(f'Val unstable ratio: {val_data["label_int"].mean():.3f}')

    data_dir = Path(cfg.data_dir)

    train_ds = StructuralDataset(train_data, data_dir, get_train_transforms(cfg.img_size))
    val_ds = StructuralDataset(val_data, data_dir, get_val_transforms(cfg.img_size))

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        num_workers=0, pin_memory=True
    )

    # Model
    model = DualStreamModel(cfg.backbone, drop_path_rate=cfg.drop_path_rate).to(device)

    # Optimizer with differential LR
    backbone_params = list(model.backbone_front.parameters()) + list(model.backbone_top.parameters())
    head_params = list(model.head.parameters())
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': cfg.lr_backbone},
        {'params': head_params, 'lr': cfg.lr_head},
    ], weight_decay=cfg.weight_decay)

    scheduler = CosineWarmupScheduler(
        optimizer, cfg.warmup_epochs, cfg.epochs, len(train_loader) // cfg.grad_accum
    )
    scaler = GradScaler()

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_path = exp_dir / f'best_fold{fold}.pt'

    for epoch in range(cfg.epochs):
        # === TRAIN ===
        model.train()
        train_losses = []
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{cfg.epochs} [Train]', leave=False)
        for step, batch in enumerate(pbar):
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            labels = batch['label'].to(device)
            soft_labels = batch['soft_label'].float().to(device)

            # Mixup / CutMix
            if random.random() < cfg.mix_prob:
                if random.random() < 0.5:
                    front, top, la, lb, sa, sb, lam = mixup_data(
                        front, top, labels, soft_labels, cfg.mixup_alpha
                    )
                else:
                    front, top, la, lb, sa, sb, lam = cutmix_data(
                        front, top, labels, soft_labels, cfg.cutmix_alpha
                    )
            else:
                la, lb, sa, sb, lam = labels, labels, soft_labels, soft_labels, 1.0

            with autocast('cuda'):
                logits = model(front, top)
                loss = compute_loss(logits, la, lb, sa, sb, lam, cfg)
                loss = loss / cfg.grad_accum

            scaler.scale(loss).backward()

            if (step + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            train_losses.append(loss.item() * cfg.grad_accum)
            pbar.set_postfix({'loss': f'{np.mean(train_losses[-20:]):.4f}'})

        # === VALIDATE ===
        model.eval()
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for batch in val_loader:
                front = batch['front'].to(device)
                top = batch['top'].to(device)

                with autocast('cuda'):
                    logits = model(front, top)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                val_preds.append(probs)
                val_labels.append(batch['label'].numpy())

        val_preds = np.concatenate(val_preds)  # (N, 2) [stable, unstable]
        val_labels = np.concatenate(val_labels)

        # LogLoss 계산 (대회 형식)
        val_true_onehot = np.zeros((len(val_labels), 2))
        val_true_onehot[np.arange(len(val_labels)), val_labels] = 1
        val_logloss = log_loss(val_labels, val_preds, labels=[0, 1])
        val_auc = roc_auc_score(val_labels, val_preds[:, 1])

        train_loss_mean = np.mean(train_losses)
        lr_current = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:2d} | TrainLoss: {train_loss_mean:.4f} | '
              f'ValLogLoss: {val_logloss:.4f} | ValAUC: {val_auc:.4f} | LR: {lr_current:.6f}')

        # Early stopping
        if val_logloss < best_val_loss:
            best_val_loss = val_logloss
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f'  -> Best model saved! LogLoss: {best_val_loss:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= cfg.early_stopping_patience:
                print(f'  -> Early stopping at epoch {epoch+1}')
                break

    # Best model로 OOF prediction
    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    model.eval()

    oof_preds = []
    with torch.no_grad():
        for batch in val_loader:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            with autocast('cuda'):
                logits = model(front, top)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            oof_preds.append(probs)

    oof_preds = np.concatenate(oof_preds)
    oof_logloss = log_loss(val_labels, oof_preds, labels=[0, 1])
    print(f'\nFold {fold} Best OOF LogLoss: {oof_logloss:.4f}')

    del model
    torch.cuda.empty_cache()

    return oof_preds, val_idx, oof_logloss

In [9]:
# 5-Fold CV 실행
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)

oof_predictions = np.zeros((len(all_df), 2))  # (1100, 2)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_df, all_df['label_int'])):
    oof_preds, val_idx_out, fold_score = train_one_fold(
        fold, train_idx, val_idx, all_df, cfg
    )
    oof_predictions[val_idx_out] = oof_preds
    fold_scores.append(fold_score)

# 전체 CV 결과
overall_logloss = log_loss(all_df['label_int'].values, oof_predictions, labels=[0, 1])
overall_auc = roc_auc_score(all_df['label_int'].values, oof_predictions[:, 1])

print(f'\n{"="*60}')
print(f'OVERALL CV RESULTS ({cfg.exp_name})')
print(f'{"="*60}')
for i, score in enumerate(fold_scores):
    print(f'  Fold {i}: LogLoss = {score:.4f}')
print(f'  Mean:   LogLoss = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'  Overall LogLoss = {overall_logloss:.4f}')
print(f'  Overall AUC     = {overall_auc:.4f}')

# OOF 예측 저장
np.save(exp_dir / 'oof_preds.npy', oof_predictions)
with open(exp_dir / 'cv_score.txt', 'w') as f:
    f.write(f'{overall_logloss:.6f}')
print(f'\nOOF predictions saved to {exp_dir / "oof_preds.npy"}')


FOLD 0
Train: 880 | Val: 220
Val unstable ratio: 0.505


Epoch  1 | TrainLoss: 0.5461 | ValLogLoss: 0.3920 | ValAUC: 0.9885 | LR: 0.000050
  -> Best model saved! LogLoss: 0.3920


Epoch  2 | TrainLoss: 0.3120 | ValLogLoss: 0.2169 | ValAUC: 1.0000 | LR: 0.000100
  -> Best model saved! LogLoss: 0.2169


Epoch  3 | TrainLoss: 0.2829 | ValLogLoss: 0.2861 | ValAUC: 0.9989 | LR: 0.000100


Epoch  4 | TrainLoss: 0.3205 | ValLogLoss: 0.2972 | ValAUC: 0.9988 | LR: 0.000099


...

Epoch 18 | TrainLoss: 0.2241 | ValLogLoss: 0.1409 | ValAUC: 1.0000 | LR: 0.000039


Epoch 19 | TrainLoss: 0.2303 | ValLogLoss: 0.1419 | ValAUC: 1.0000 | LR: 0.000033


Epoch 20 | TrainLoss: 0.2291 | ValLogLoss: 0.1501 | ValAUC: 1.0000 | LR: 0.000028


Epoch 21 | TrainLoss: 0.2296 | ValLogLoss: 0.1534 | ValAUC: 1.0000 | LR: 0.000023


Epoch 22 | TrainLoss: 0.2457 | ValLogLoss: 0.1411 | ValAUC: 1.0000 | LR: 0.000019
  -> Early stopping at epoch 22

Fold 4 Best OOF LogLoss: 0.1366

OVERALL CV RESULTS (convnextv2_base)
  Fold 0: LogLoss = 0.1232
  Fold 1: LogLoss = 0.1332
  Fold 2: LogLoss = 0.1335
  Fold 3: LogLoss = 0.1216
  Fold 4: LogLoss = 0.1366
  Mean:   LogLoss = 0.1296 ± 0.0060
  Overall LogLoss = 0.1296
  Overall AUC     = 0.9982

OOF predictions saved to ..\outputs\convnextv2_base\oof_preds.npy


## 5. Test Inference + TTA

In [10]:
def predict_test(cfg, tta_count=5):
    """5-Fold 모델로 Test 예측 (TTA 포함)"""
    data_dir = Path(cfg.data_dir)
    test_df_local = pd.read_csv(data_dir / 'sample_submission.csv')

    all_preds = []  # fold별 예측 수집

    for fold in range(cfg.n_folds):
        print(f'Predicting with fold {fold} model...')
        model_path = exp_dir / f'best_fold{fold}.pt'
        model = DualStreamModel(cfg.backbone, drop_path_rate=0.0).to(device)
        model.load_state_dict(torch.load(model_path, weights_only=True))
        model.eval()

        fold_preds = []

        for tta_idx in range(tta_count):
            if tta_idx == 0:
                transforms = get_val_transforms(cfg.img_size)
            else:
                transforms = get_tta_transforms(cfg.img_size)

            test_ds = StructuralDataset(
                test_df_local, data_dir, transforms, is_test=True
            )
            test_loader = DataLoader(
                test_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                num_workers=0, pin_memory=True
            )

            tta_preds = []
            with torch.no_grad():
                for batch in test_loader:
                    front = batch['front'].to(device)
                    top = batch['top'].to(device)
                    with autocast('cuda'):
                        logits = model(front, top)
                    probs = F.softmax(logits, dim=1).cpu().numpy()
                    tta_preds.append(probs)

            tta_preds = np.concatenate(tta_preds)
            fold_preds.append(tta_preds)

        # TTA 평균
        fold_mean = np.mean(fold_preds, axis=0)
        all_preds.append(fold_mean)

        del model
        torch.cuda.empty_cache()

    # 5-Fold 평균
    test_preds = np.mean(all_preds, axis=0)  # (1000, 2)
    print(f'Test predictions shape: {test_preds.shape}')
    print(f'Unstable prob range: [{test_preds[:, 1].min():.4f}, {test_preds[:, 1].max():.4f}]')

    return test_preds

test_preds = predict_test(cfg, tta_count=cfg.tta_count)
np.save(exp_dir / 'test_preds.npy', test_preds)
print(f'Test predictions saved to {exp_dir / "test_preds.npy"}')

Predicting with fold 0 model...


Predicting with fold 1 model...
Predicting with fold 2 model...
Predicting with fold 3 model...
Predicting with fold 4 model...
Test predictions shape: (1000, 2)
Unstable prob range: [0.1080, 0.9009]
Test predictions saved to ..\outputs\convnextv2_base\test_preds.npy


## 6. 제출 파일 생성 (단일 모델)

In [23]:
# 단일 모델 제출 파일
data_dir = Path(cfg.data_dir)
test_df_sub = pd.read_csv(data_dir / 'sample_submission.csv')

# float64로 변환 (float16 비교 문제 방지)
unstable_prob = test_preds[:, 1].astype(np.float64)
stable_prob = test_preds[:, 0].astype(np.float64)

# unstable_prob 기준으로 stable_prob 계산 → 합이 정확히 1
unstable_prob = np.clip(unstable_prob, 1e-15, 1 - 1e-15)
stable_prob = 1.0 - unstable_prob

submission = pd.DataFrame({
    'id': test_df_sub['id'],
    'unstable_prob': unstable_prob,
    'stable_prob': stable_prob,
})

# 검증
assert (submission[['unstable_prob', 'stable_prob']].sum(axis=1) - 1.0).abs().max() < 1e-10
assert (submission['unstable_prob'] >= 0).all()
assert (submission['stable_prob'] >= 0).all()

sub_path = Path('../submissions') / f'{cfg.exp_name}_submission.csv'
sub_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(sub_path, encoding='UTF-8-sig', index=False)
print(f'Submission saved: {sub_path}')
print(f'Shape: {submission.shape}')
submission.head(10)

Submission saved: ..\submissions\convnextv2_base_submission.csv
Shape: (1000, 3)


,id,unstable_prob,stable_prob
0,TEST_0001,0.109924,0.890076
1,TEST_0002,0.887695,0.112305
2,TEST_0003,0.887695,0.112305
3,TEST_0004,0.886719,0.113281
4,TEST_0005,0.222412,0.777588
5,TEST_0006,0.890137,0.109863
6,TEST_0007,0.137817,0.862183
7,TEST_0008,0.896484,0.103516
8,TEST_0009,0.890625,0.109375
9,TEST_0010,0.139893,0.860107


---

## 7. 앙상블 (다중 backbone 학습 후)

**사용법:** 위의 Config에서 backbone/exp_name만 바꿔서 전체 노트북을 다시 실행.
여러 모델의 OOF + Test 예측이 `outputs/` 에 쌓이면 이 셀에서 앙상블.

```python
# 2번째 실행 예시:
cfg = Config(
    backbone='efficientnetv2_rw_s.ra2_in1k',
    exp_name='efficientnetv2_s',
    img_size=384,
)
```

In [ ]:
# 앙상블: outputs/ 하위의 모든 모델 예측을 로딩하여 최적 가중치 탐색
from scipy.optimize import minimize

output_root = Path('../outputs')
model_dirs = [d for d in output_root.iterdir() if d.is_dir() and (d / 'oof_preds.npy').exists()]

if len(model_dirs) < 2:
    print(f'앙상블 불가: {len(model_dirs)}개 모델만 있음. backbone 변경 후 재학습 필요.')
    print(f'현재 모델: {[d.name for d in model_dirs]}')
else:
    print(f'앙상블 대상 모델: {[d.name for d in model_dirs]}')

    # OOF 예측 로딩
    oof_list = []
    test_list = []
    names = []
    for d in model_dirs:
        oof = np.load(d / 'oof_preds.npy').astype(np.float64)
        test = np.load(d / 'test_preds.npy').astype(np.float64)
        cv_score = float(open(d / 'cv_score.txt').read().strip())
        oof_list.append(oof)
        test_list.append(test)
        names.append(d.name)
        print(f'  {d.name}: CV LogLoss = {cv_score:.4f}')

    y_true = all_df['label_int'].values

    # 단일 모델 상관관계
    print('\n=== Model Correlation (unstable_prob) ===')
    corr_matrix = np.corrcoef([o[:, 1] for o in oof_list])
    for i, n1 in enumerate(names):
        for j, n2 in enumerate(names):
            if i < j:
                print(f'  {n1} vs {n2}: {corr_matrix[i,j]:.4f}')

    # 최적 가중치 탐색
    def ensemble_logloss(weights):
        weights = np.abs(weights)
        weights = weights / weights.sum()
        blended = np.zeros_like(oof_list[0])
        for w, oof in zip(weights, oof_list):
            blended += w * oof
        return log_loss(y_true, blended, labels=[0, 1])

    n_models = len(oof_list)
    init_weights = np.ones(n_models) / n_models
    result = minimize(ensemble_logloss, init_weights, method='Nelder-Mead')
    best_weights = np.abs(result.x)
    best_weights = best_weights / best_weights.sum()

    print(f'\n=== Optimal Weights ===')
    for n, w in zip(names, best_weights):
        print(f'  {n}: {w:.4f}')

    # 앙상블 CV 점수
    ensemble_oof = sum(w * o for w, o in zip(best_weights, oof_list))
    ens_logloss = log_loss(y_true, ensemble_oof, labels=[0, 1])
    ens_auc = roc_auc_score(y_true, ensemble_oof[:, 1])
    print(f'\nEnsemble CV LogLoss: {ens_logloss:.4f}')
    print(f'Ensemble CV AUC:     {ens_auc:.4f}')

    # Temperature Scaling
    from scipy.optimize import minimize_scalar

    def temp_logloss(T):
        scaled = np.exp(np.log(np.clip(ensemble_oof, 1e-15, 1)) / T)
        scaled = scaled / scaled.sum(axis=1, keepdims=True)
        return log_loss(y_true, scaled, labels=[0, 1])

    temp_result = minimize_scalar(temp_logloss, bounds=(0.5, 5.0), method='bounded')
    best_T = temp_result.x
    print(f'\nOptimal Temperature: {best_T:.4f}')
    print(f'After Temp Scaling LogLoss: {temp_result.fun:.4f}')

    # 최종 Test 예측
    ensemble_test = sum(w * t for w, t in zip(best_weights, test_list))
    if abs(best_T - 1.0) > 0.01:
        ensemble_test = np.exp(np.log(np.clip(ensemble_test, 1e-15, 1)) / best_T)
        ensemble_test = ensemble_test / ensemble_test.sum(axis=1, keepdims=True)

    # 앙상블 제출 파일 (float64 + stable = 1 - unstable)
    test_df_sub = pd.read_csv(Path(cfg.data_dir) / 'sample_submission.csv')
    ens_unstable = np.clip(ensemble_test[:, 1].astype(np.float64), 1e-15, 1 - 1e-15)
    ens_submission = pd.DataFrame({
        'id': test_df_sub['id'],
        'unstable_prob': ens_unstable,
        'stable_prob': 1.0 - ens_unstable,
    })

    ens_path = Path('../submissions') / 'ensemble_submission.csv'
    ens_submission.to_csv(ens_path, encoding='UTF-8-sig', index=False)
    print(f'\nEnsemble submission saved: {ens_path}')
    ens_submission.head(10)

## 사용 가이드

### 1차 실행 (ConvNeXt-V2-Base)
- 그대로 전체 실행 → `outputs/convnextv2_base/` + `submissions/convnextv2_base_submission.csv`

### 2차 실행 (다른 backbone)
Config 셀에서 아래만 변경 후 전체 재실행:

```python
cfg = Config(
    backbone='efficientnetv2_rw_s.ra2_in1k',
    exp_name='efficientnetv2_s',
)
```

### 3차 실행 이후: 추천 backbone
| exp_name | backbone | img_size |
|----------|----------|----------|
| efficientnetv2_s | efficientnetv2_rw_s.ra2_in1k | 384 |
| swinv2_base | swinv2_base_window12to16_192to256.ms_in22k_ft_in1k_256 | 256 |
| eva02_small | eva02_small_patch14_336.mim_in22k_ft_in1k | 336 |
| convnext_clip | convnext_base.clip_laion2b_augreg_ft_in12k_in1k_384 | 384 |

### 앙상블
- 2개 이상 모델이 `outputs/`에 있으면 Section 7 앙상블 셀 실행
- → `submissions/ensemble_submission.csv` 생성